In [ ]:
"""
=============================================================================
Beyond Accuracy: A Reliability-Oriented Framework for Brain Tumor MRI Classification with Calibration, 
Explainability, and Cross-Dataset Validation
=============================================================================
Authors: Ibrahim Hayatu Hassan
"""

import os
import json
import cv2
import gc
import time
import random
import numpy as np
import pandas as pd
import sys
from typing import Any, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import chi2
from collections import Counter, defaultdict
from pathlib import Path
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch import amp # Unified AMP API
import torchvision.transforms as transforms
import timm
from statsmodels.stats.contingency_tables import mcnemar as smm_mcnemar
from statsmodels.stats.multitest import multipletests

try:
    import thop
    print("thop found")
except ImportError:
    !pip install thop --no-deps -q
    import thop
    print("thop installed")

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, auc as sk_auc, cohen_kappa_score
)
import shap
import warnings

warnings.filterwarnings("ignore")

# ═════════════════════════════════════════════════════════════════════════════
# === REVISION SETUP (set these before every run) =============================
# ═════════════════════════════════════════════════════════════════════════════
# These two variables are the ONLY things you change between runs:
#   DATASET = "D1" | "D2" | "D3"|D4
#   SMOKE_TEST = True  → ~10-min sanity run (5 epochs, 1 CV fold, no baselines)
#                False → full canonical run (~12h on Kaggle T4)
#
# All other behaviour is derived. Outputs go to ./outputs_pytorch/<DATASET>/
# so the four runs never overwrite each other.
# ═════════════════════════════════════════════════════════════════════════════
DATASET    = "D2"          # change to D1, D2, D3, or D4 before each run
SMOKE_TEST = False          # set False for the real canonical run

# Import the helper module that bundles all revisions.
# This file must live in the same directory as the notebook
# (on Kaggle: upload beyond_accuracy.ipy via Add Data → Upload, or paste it into a
#  separate cell above this one).

# Module-level collection bins. The ablation, baseline, and XAI cells append
# their results to these lists/dicts so main() can flush them into the JSON
# summary at the end. Using globals keeps the change minimal — none of the
# existing function signatures change. Comment 2 root-cause fix.
_ABLATION_RESULTS: list = []
_BASELINE_RESULTS: list = []
_MCNEMAR_ROWS_ABLATION: list = []
_MCNEMAR_ROWS_BASELINE: list = []
_BASELINE_BOOTSTRAP_CIS: dict = {}
_BASELINE_PREDICTIONS: dict = {}
_XAI_IOU_PER_CLASS: dict = {}
_XAI_DEL_INS_AUC: dict = {}

    
# ImageNet Normalization Constants
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# ─────────────────────────────────────────────────────────────────────────────
# 0. Global Reproducibility
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────────────────────────────────────
# 0.5 Logger Suite
# ─────────────────────────────────────────────────────────────────────────────
class Logger(object):
    def __init__(self, filename="log.txt"):
        self.terminal = sys.stdout
        self.error = sys.stderr
        self.log = open(filename, "a", encoding="utf-8")
        sys.stdout = self
        sys.stderr = self

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# ============================================================================
# Dataset Configuration Registry
# ============================================================================

DATASET_CONFIGS = {
    "D1": {
        "name": "Kaggle (SARTAJ)",
        "candidate_dataset_dirs": [
            "/kaggle/input/brain-tumor-classification-mri",
            "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri",
        ],
        "num_classes": 4,
        "class_folders": [
            "glioma_tumor",
            "meningioma_tumor",
            "no_tumor",
            "pituitary_tumor"
        ],
        "class_labels": [
            "glioma",
            "meningioma",
            "no_tumor",
            "pituitary"
        ],
        "output_subdir": "D1",
        "has_training_testing_subdirs": True,
    },

    "D2": {
        "name": "BRISC-2025",
        "candidate_dataset_dirs": [
            "/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task",
            "/kaggle/input/brisc2025",
            "/kaggle/input/brisc-2025",
        ],
        "num_classes": 4,
        "class_folders": [
            "glioma",
            "meningioma",
            "no_tumor",
            "pituitary"
        ],
        "class_labels": [
            "glioma",
            "meningioma",
            "no_tumor",
            "pituitary"
        ],
        "output_subdir": "D2",
        "has_training_testing_subdirs": True,
    },

    "D3": {
        "name": "Figshare (Cheng 2017)",
        "candidate_dataset_dirs": [
            "/kaggle/input/datasets/denizkavi1/brain-tumor",
            "/kaggle/input/brain-tumor",
            "/kaggle/input/figshare-brain-tumor-dataset",
            "/kaggle/input/figshare-brain-tumor-mri",
        ],
        "num_classes": 3,
        "class_folders": ["2", "1", "3"],
        "class_labels": [
            "glioma",
            "meningioma",
            "pituitary"
        ],
        "output_subdir": "D3",
        "has_training_testing_subdirs": False,
    },
     
     "D4": {
        "name": "Nickparvar (Combined)",
        "candidate_dataset_dirs": [
            "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset",
            "/kaggle/input/brain-tumor-mri-dataset",
        ],
        "num_classes": 4,
        "class_folders": [
            "glioma",
            "meningioma",
            "notumor",
            "pituitary"
        ],
        "class_labels": [
            "glioma",
            "meningioma",
            "no_tumor",
            "pituitary"
        ],
        "output_subdir": "D4",
        "has_training_testing_subdirs": True,
    },
}

def apply_dataset_config(Config_cls, dataset_key):
    if dataset_key not in DATASET_CONFIGS:
        raise KeyError(
            f"Unknown DATASET={dataset_key}. Choose from {list(DATASET_CONFIGS.keys())}"
        )

    cfg = DATASET_CONFIGS[dataset_key]

    chosen_dir = None
    for cand in cfg["candidate_dataset_dirs"]:
        if os.path.exists(cand):
            chosen_dir = cand
            break

    if chosen_dir is None:
        chosen_dir = cfg["candidate_dataset_dirs"][0]
        print(f"[WARN] No candidate path exists for {dataset_key}. Using: {chosen_dir}")

    Config_cls.DATASET_DIR = chosen_dir
    Config_cls.TRAIN_DIR = os.path.join(chosen_dir, "Training")
    Config_cls.TEST_DIR = os.path.join(chosen_dir, "Testing")
    Config_cls.NUM_CLASSES = cfg["num_classes"]
    Config_cls.CLASSES = cfg["class_folders"]
    Config_cls.CLASS_LABELS = cfg["class_labels"]
    Config_cls.OUTPUT_DIR = os.path.join("./outputs_pytorch", cfg["output_subdir"])
    Config_cls.CHECKPOINT = os.path.join(
        Config_cls.OUTPUT_DIR,
        "best_effnetv2_bt_pytorch.pth"
    )

    os.makedirs(Config_cls.OUTPUT_DIR, exist_ok=True)

    print(f"[CONFIG] DATASET = {dataset_key} ({cfg['name']})")
    print(f"[CONFIG] DATASET_DIR = {Config_cls.DATASET_DIR}")
    print(f"[CONFIG] NUM_CLASSES = {Config_cls.NUM_CLASSES}")
    print(f"[CONFIG] CLASSES = {Config_cls.CLASSES}")
    print(f"[CONFIG] CLASS_LABELS = {Config_cls.CLASS_LABELS}")
    print(f"[CONFIG] OUTPUT_DIR = {Config_cls.OUTPUT_DIR}")

    return cfg
# ─────────────────────────────────────────────────────────────────────────────
# 1. Configuration
# ─────────────────────────────────────────────────────────────────────────────
class Config:
    DATASET_DIR  = " "
  
    DATASET_DIR = ""
    TRAIN_DIR = ""
    TEST_DIR = ""

    OUTPUT_DIR = "./outputs_pytorch"
    CHECKPOINT = os.path.join(
        OUTPUT_DIR,
        "best_effnetv2_bt_pytorch.pth"
    )
    
    IMG_SIZE     = (224, 224)
    CHANNELS     = 3
    NUM_CLASSES  = 4
    CLASSES      = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
    CLASS_LABELS = ["glioma", "meningioma", "no tumor", "pituitary"]

    VAL_SPLIT    = 0.10
    TEST_SPLIT   = 0.10
    BATCH_SIZE   = 32
    EPOCHS       = 100
    WARMUP_EPOCHS = 15   # Increased from 10: dual-pooling attention head needs more warmup to stabilise
    CV_EPOCHS    = 100
    CV_WARMUP_EPOCHS = 15
    BASELINE_EPOCHS = 100
    PATIENCE     = 20

    LR_HEAD      = 1e-4
    LR_FT        = 2e-5
    WEIGHT_DECAY = 1e-4
    DROPOUT      = 0.35  # Reduced from 0.5: lower dropout better suited to EfficientNet-scale models
    MEDIAN_K     = 5

    # Progressive Resolution
    PROGRESSIVE_RES = True
    RES_SCHEDULE = [(0, 160), (30, 224)] # (epoch, size)

    # Augmentation Parity
    AUG_ROTATION = 10
    AUG_BRIGHT   = 0.1
    AUG_WIDTH    = 0.05
    AUG_HEIGHT   = 0.05
    AUG_ZOOM     = 0.05
    # Augmentations
    AUG_HFLIP    = False
    LABEL_SMOOTHING = 0.05  # Reduced from 0.1: less aggressive smoothing for clean 4-class labels

    # Metrics settings
    BOOTSTRAP_B    = 2000
    XAI_TOP_K_PCT  = 0.10
    ECE_BINS       = 15
    DI_STEPS       = 100
    N_FOLDS        = 5
    SHAP_BG_SIZE   = 50

    # SWA
    USE_SWA = True
    SWA_START_EPOCH = 75
    
    # Ablation Settings
    ABLATION_EPOCHS = 100
    ABLATION_HEAD_EPOCHS = 15
    ABLATION_FT_EPOCHS = 100

_dataset_cfg_entry = apply_dataset_config(
    Config,
    DATASET
)

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

if SMOKE_TEST:
    # Smoke test: short, cheap run to verify the pipeline works end-to-end.
    Config.EPOCHS = 5
    Config.WARMUP_EPOCHS = 2
    Config.CV_EPOCHS = 5
    Config.CV_WARMUP_EPOCHS = 2
    Config.BASELINE_EPOCHS = 5
    Config.ABLATION_EPOCHS = 5
    Config.ABLATION_HEAD_EPOCHS = 2
    Config.N_FOLDS = 2               # minimum for StratifiedKFold; CV is essentially skipped in smoke
    Config.BOOTSTRAP_B = 100         # 100 instead of 2000
    Config.PATIENCE = 99             # disable early stop in smoke (too few epochs)
    Config.SWA_START_EPOCH = max(1, int(0.75 * Config.EPOCHS))
    print(f"  [SMOKE] Smoke test mode: {Config.EPOCHS} epochs, "
          f"{Config.N_FOLDS}-fold CV, baselines @ {Config.BASELINE_EPOCHS} epochs (Holm-corrected McNemar will run)")
else:
    print(f"  [RUN] Full canonical run: {Config.EPOCHS} epochs, "
          f"{Config.N_FOLDS}-fold CV, all baselines")


# Dataset validation check (dataset-aware)
# The data loader supports three layouts: Training/Testing/, train/test/, or flat.
# We just verify the top-level DATASET_DIR exists; the loader emits a clear error
# if no class folders are found inside.
if not os.path.exists(Config.DATASET_DIR):
    print(f"\n  [ERROR] DATASET_DIR does not exist: {Config.DATASET_DIR}")
    print(f"  Make sure the Kaggle dataset for {DATASET} is attached to this notebook.")
    # We don't exit so users can still run small offline tests.


# ─────────────────────────────────────────────────────────────────────────────
# 2. Advanced Preprocessing
# ─────────────────────────────────────────────────────────────────────────────
def crop_brain_contour(img):
    """Detects and crops the brain area to remove black borders."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)
    cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    if not cnts:
        return img
    c = max(cnts, key=cv2.contourArea)
    left, right = c[:, :, 0].min(), c[:, :, 0].max()
    top, bottom = c[:, :, 1].min(), c[:, :, 1].max()
    h, w, _ = img.shape
    return img[max(0, top-5):min(h, bottom+5), max(0, left-5):min(w, right+5)]

def _load_and_enhance(img_path: str) -> np.ndarray:
    img = cv2.imread(img_path)
    if img is None: raise FileNotFoundError(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_brain_contour(img)
    img = cv2.resize(img, Config.IMG_SIZE, interpolation=cv2.INTER_LINEAR)
    img = cv2.medianBlur(img, Config.MEDIAN_K)
    return img

# ─────────────────────────────────────────────────────────────────────────────
# 3. PyTorch Dataset & Dataloaders
# ─────────────────────────────────────────────────────────────────────────────
class BrainTumorDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
        else:
            # Default fallback: normalize manually if no transform provided
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
            
        return img, label

def get_transforms(img_size=Config.IMG_SIZE[0], augment=False):
    if augment:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.RandomAffine(
                degrees=Config.AUG_ROTATION,
                translate=(Config.AUG_WIDTH, Config.AUG_HEIGHT),
                scale=(1 - Config.AUG_ZOOM, 1 + Config.AUG_ZOOM)
            ),
            transforms.ColorJitter(brightness=Config.AUG_BRIGHT),
            transforms.RandomHorizontalFlip(p=0.5 if Config.AUG_HFLIP else 0.0),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])
    else:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])

def _phash_from_path(img_path: str, hash_size: int = 8, highfreq_factor: int = 4) -> int:
    """
    DCT-based perceptual hash computed from the original image file.
    This is used only for leakage screening, before any train/val/cal/test split.
    """
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    size = hash_size * highfreq_factor
    gray = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(gray)
    dct_low = dct[:hash_size, :hash_size]
    med = np.median(dct_low[1:, 1:])  # exclude DC term
    bits = (dct_low > med).astype(np.uint8).flatten()
    h = 0
    for bit in bits:
        h = (h << 1) | int(bit)
    return int(h)


def _hamming_int(a: int, b: int) -> int:
    """Hamming distance between two integer hashes."""
    return int((int(a) ^ int(b)).bit_count())


def _build_phash_duplicate_groups(records, threshold: int = 6, hash_size: int = 8):
    """
    Build duplicate/near-duplicate groups before splitting.

    Exact duplicates: Hamming distance = 0
    Near-duplicate candidates: Hamming distance <= threshold

    The output group id is later used as a split constraint, so all members of the
    same duplicate group are assigned to the same train/cal/val/test partition.
    """
    n = len(records)
    print(f"  [DUP] Computing {hash_size*hash_size}-bit pHash for {n} images...")

    for r in tqdm(records, desc="  pHash"):
        r["phash"] = _phash_from_path(r["path"], hash_size=hash_size)

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    exact_pairs = []
    near_pairs = []

    print(f"  [DUP] Comparing pHash values across the full image pool (threshold <= {threshold})...")
    hashes = [r["phash"] for r in records]
    for i in tqdm(range(n), desc="  pHash compare"):
        hi = hashes[i]
        for j in range(i + 1, n):
            d = _hamming_int(hi, hashes[j])
            if d == 0:
                exact_pairs.append((i, j, d))
                union(i, j)
            elif d <= threshold:
                near_pairs.append((i, j, d))
                union(i, j)

    roots = [find(i) for i in range(n)]
    root_to_gid = {root: gid for gid, root in enumerate(sorted(set(roots)))}
    for i, r in enumerate(records):
        r["dup_group"] = root_to_gid[roots[i]]

    return records, exact_pairs, near_pairs


def _stratified_group_split_records(records, seed: int = SEED):
    """
    Stratified group-level split.
    Each duplicate group appears in only one split. Stratification is performed
    on group-level majority labels, which is appropriate because most groups are
    singletons and duplicate groups should not be split.
    """
    import pandas as pd

    df = pd.DataFrame(records)
    group_df = (
        df.groupby("dup_group")
          .agg(label=("label", lambda x: Counter(x).most_common(1)[0][0]),
               n_images=("path", "count"))
          .reset_index()
    )

    # test = 10% of groups, stratified by group-level majority class
    g_tv, g_test = train_test_split(
        group_df,
        test_size=Config.TEST_SPLIT,
        stratify=group_df["label"],
        random_state=seed,
    )

    # val = 10% of full dataset -> relative fraction from remaining groups
    val_relative = Config.VAL_SPLIT / (1.0 - Config.TEST_SPLIT)
    g_train_full, g_val = train_test_split(
        g_tv,
        test_size=val_relative,
        stratify=g_tv["label"],
        random_state=seed,
    )

    # calibration = 10% of remaining training groups, dedicated to temperature scaling
    g_train, g_cal = train_test_split(
        g_train_full,
        test_size=0.10,
        stratify=g_train_full["label"],
        random_state=seed,
    )

    split_map = {}
    for g in g_train["dup_group"].tolist(): split_map[g] = "train"
    for g in g_cal["dup_group"].tolist():   split_map[g] = "cal"
    for g in g_val["dup_group"].tolist():   split_map[g] = "val"
    for g in g_test["dup_group"].tolist():  split_map[g] = "test"

    df["split"] = df["dup_group"].map(split_map)

    leaked = df.groupby("dup_group")["split"].nunique()
    leaked = leaked[leaked > 1]
    if len(leaked) > 0:
        raise RuntimeError("Duplicate groups were split across partitions. This should never happen.")

    return df


def _save_dedup_report(df, exact_pairs, near_pairs, threshold: int = 6, hash_size: int = 8):
    """Save an auditable CSV/JSON report for reviewers and supplementary material."""
    import json

    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
    split_csv = os.path.join(Config.OUTPUT_DIR, f"{DATASET}_phash_group_split.csv")
    report_json = os.path.join(Config.OUTPUT_DIR, f"{DATASET}_phash_dedup_report.json")
    pair_csv = os.path.join(Config.OUTPUT_DIR, f"{DATASET}_phash_candidate_pairs.csv")

    df.to_csv(split_csv, index=False)

    # Write candidate pairs with paths for audit/manual inspection.
    pair_rows = []
    records = df.reset_index(drop=True).to_dict("records")
    for i, j, d in exact_pairs:
        pair_rows.append({
            "kind": "exact", "hamming": int(d),
            "path_a": records[i]["path"], "label_a": records[i]["label"],
            "path_b": records[j]["path"], "label_b": records[j]["label"],
        })
    for i, j, d in near_pairs:
        pair_rows.append({
            "kind": "near", "hamming": int(d),
            "path_a": records[i]["path"], "label_a": records[i]["label"],
            "path_b": records[j]["path"], "label_b": records[j]["label"],
        })
    pd.DataFrame(pair_rows).to_csv(pair_csv, index=False)

    group_sizes = df.groupby("dup_group").size()
    report = {
        "dataset": DATASET,
        "hash_type": "DCT perceptual hash",
        "hash_bits": int(hash_size * hash_size),
        "near_duplicate_hamming_threshold": int(threshold),
        "total_images": int(len(df)),
        "unique_duplicate_groups": int(df["dup_group"].nunique()),
        "non_singleton_duplicate_groups": int((group_sizes > 1).sum()),
        "largest_duplicate_group_size": int(group_sizes.max()),
        "exact_phash_pairs": int(len(exact_pairs)),
        "near_duplicate_candidate_pairs": int(len(near_pairs)),
        "split_counts": {str(k): int(v) for k, v in df["split"].value_counts().sort_index().items()},
        "class_by_split": df.groupby(["split", "label"]).size().unstack(fill_value=0).astype(int).to_dict(),
        "leakage_check": "PASS: no pHash duplicate group crosses train/cal/val/test partitions",
        "split_csv": split_csv,
        "candidate_pair_csv": pair_csv,
    }

    with open(report_json, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

    print("\n  [DUP] pHash duplicate-screening report")
    print("  " + "-" * 66)
    print(f"  Total images:                  {report['total_images']}")
    print(f"  Unique pHash groups:           {report['unique_duplicate_groups']}")
    print(f"  Non-singleton groups:          {report['non_singleton_duplicate_groups']}")
    print(f"  Exact duplicate pairs:         {report['exact_phash_pairs']}")
    print(f"  Near-duplicate candidate pairs:{report['near_duplicate_candidate_pairs']}")
    print(f"  Largest group size:            {report['largest_duplicate_group_size']}")
    print(f"  Split counts:                  {report['split_counts']}")
    print(f"  Leakage check:                 {report['leakage_check']}")
    print(f"  Saved split CSV:               {split_csv}")
    print(f"  Saved pair CSV:                {pair_csv}")
    print(f"  Saved JSON report:             {report_json}")
    print("  " + "-" * 66 + "\n")

    return report


def load_dataset_unified_pytorch(train_dir: str, test_dir: str):
    """
    Leakage-aware data loader.

    Correct reviewer-safe order:
      1. scan all released image paths from the public dataset;
      2. compute pHash before splitting;
      3. group exact/near-duplicate images;
      4. assign each duplicate group to only one of train/cal/val/test;
      5. load and preprocess images after split assignment.

    This avoids the invalid practice of removing duplicates only from the test set
    after model training. It also produces CSV/JSON audit files for the manuscript
    supplementary material.
    """
    print(f"  [LOAD] DATASET_DIR = {Config.DATASET_DIR}")
    records = []
    found_any = False

    def _scan_layout_a_or_b(root, train_name, test_name):
        found = False
        for d_name in [train_name, test_name]:
            d = os.path.join(root, d_name)
            if not os.path.isdir(d):
                continue
            for cls in Config.CLASSES:
                cls_p = os.path.join(d, cls)
                if not os.path.isdir(cls_p):
                    continue
                files = sorted([f for f in os.listdir(cls_p) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
                if not files:
                    continue
                found = True
                for fn in files:
                    records.append({
                        "path": os.path.join(cls_p, fn),
                        "filename": fn,
                        "source_split": d_name,
                        "disk_class": cls,
                        "label": Config.CLASS_LABELS[Config.CLASSES.index(cls)],
                    })
        return found

    def _scan_layout_c(root):
        found = False
        for cls in Config.CLASSES:
            cls_p = os.path.join(root, cls)
            if not os.path.isdir(cls_p):
                continue
            files = sorted([f for f in os.listdir(cls_p) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            if not files:
                continue
            found = True
            for fn in files:
                records.append({
                    "path": os.path.join(cls_p, fn),
                    "filename": fn,
                    "source_split": "flat",
                    "disk_class": cls,
                    "label": Config.CLASS_LABELS[Config.CLASSES.index(cls)],
                })
        return found

    # Try common layouts in order.
    for tr_name, te_name in [("Training", "Testing"), ("train", "test")]:
        if _scan_layout_a_or_b(Config.DATASET_DIR, tr_name, te_name):
            found_any = True
            print(f"  [LOAD] Detected layout: {tr_name}/{te_name}/<class>/")
            break
    if not found_any:
        if _scan_layout_c(Config.DATASET_DIR):
            found_any = True
            print("  [LOAD] Detected layout: <class>/ (flat)")

    if not found_any:
        try:
            entries = os.listdir(Config.DATASET_DIR)[:20]
        except FileNotFoundError:
            entries = ["(directory does not exist)"]
        raise FileNotFoundError(
            f"No images found under {Config.DATASET_DIR}.\n"
            f"Tried layouts: Training/Testing, train/test, flat.\n"
            f"Expected class folders: {Config.CLASSES}\n"
            f"Top-level entries actually present: {entries}\n"
            "Check that the Kaggle dataset is attached to this notebook."
        )

    print(f"  [LOAD] Found {len(records)} images before pHash screening.")
    print(f"  [LOAD] Class counts before split: {dict(Counter([r['label'] for r in records]))}")

    records, exact_pairs, near_pairs = _build_phash_duplicate_groups(
        records, threshold=6, hash_size=8
    )
    df = _stratified_group_split_records(records, seed=SEED)
    _save_dedup_report(df, exact_pairs, near_pairs, threshold=6, hash_size=8)

    # Load enhanced image arrays only after leakage-safe split assignment.
    le = LabelEncoder().fit(Config.CLASS_LABELS)

    def _load_split(split_name):
        sub = df[df["split"] == split_name].reset_index(drop=True)
        imgs, labs = [], []
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"  Loading {split_name}"):
            try:
                imgs.append(_load_and_enhance(row["path"]))
                labs.append(row["label"])
            except Exception as e:
                print(f"  [WARN] Failed to load {row['path']}: {e}")
        X = np.array(imgs, dtype=np.uint8)
        y = le.transform(np.array(labs))
        return X, y

    X_train, y_train = _load_split("train")
    X_cal, y_cal = _load_split("cal")
    X_val, y_val = _load_split("val")
    X_test, y_test = _load_split("test")

    print(f"  [RESULT] Leakage-safe group split -> Train:{len(X_train)}  Cal:{len(X_cal)}  Val:{len(X_val)}  Test:{len(X_test)}")
    print(f"  [RESULT] Train class counts: {dict(zip(le.classes_, np.bincount(y_train, minlength=len(le.classes_))))}")
    print(f"  [RESULT] Cal class counts:   {dict(zip(le.classes_, np.bincount(y_cal, minlength=len(le.classes_))))}")
    print(f"  [RESULT] Val class counts:   {dict(zip(le.classes_, np.bincount(y_val, minlength=len(le.classes_))))}")
    print(f"  [RESULT] Test class counts:  {dict(zip(le.classes_, np.bincount(y_test, minlength=len(le.classes_))))}")

    return (X_train, y_train, X_cal, y_cal, X_val, y_val, X_test, y_test, le)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Models
# ─────────────────────────────────────────────────────────────────────────────
class DualPoolingAttention(nn.Module):
    """
    Dual-pooling attention with FP32-stable computation (Issue 2 fix).
    The attention softmax is explicitly computed in float32 to avoid
    FP16 underflow under mixed precision training.
    """
    def __init__(self, in_channels):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)
        # Keep attention linear in float32 always
        self.attn_fc = nn.Linear(in_channels, 1, bias=True).float()

    def forward(self, x):
        gap = self.gap(x).view(x.size(0), 1, -1)  # (B, 1, C)
        gmp = self.gmp(x).view(x.size(0), 1, -1)  # (B, 1, C)
        concat = torch.cat([gap, gmp], dim=1)       # (B, 2, C)

        # Compute attention weights in float32 for numerical stability
        concat_fp32 = concat.float()                       # (B, 2, C)
        attn_logits = self.attn_fc(concat_fp32)            # (B, 2, 1)
        attn_weights = torch.softmax(attn_logits, dim=1)  # (B, 2, 1) — stable

        # Cast back to input dtype for the weighted sum
        attn_weights = attn_weights.to(concat.dtype)
        weighted = (concat * attn_weights).sum(dim=1)     # (B, C)
        return weighted

class BrainTumorModel(nn.Module):
    def __init__(self, num_classes=Config.NUM_CLASSES, dropout=Config.DROPOUT, use_attention=True):
        super().__init__()
        # Using timm for EfficientNetV2-B0
        self.backbone = timm.create_model('tf_efficientnetv2_b0.in1k', pretrained=True, num_classes=0, global_pool='')
        self.use_attention = use_attention
        
        # Get feature dimension dynamically
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            feat_dim = self.backbone(dummy).shape[1]
            
        if use_attention:
            self.pool_head = DualPoolingAttention(feat_dim)
        else:
            self.pool_head = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten()
            )

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 512),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x) # (batch, channels, h, w)
        pooled = self.pool_head(features)
        logits = self.classifier(pooled)
        return logits

# ─────────────────────────────────────────────────────────────────────────────
# 5. Training Logic
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStoppingPyTorch:
    def __init__(self, patience=Config.PATIENCE, verbose=False, path=Config.CHECKPOINT):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...")
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        with amp.autocast(device_type=device.type, enabled=(device.type != 'cpu')):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix(loss=loss.item(), acc=100.*correct/total)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    return running_loss/total, correct/total

def train_model_pytorch(model, X_train, y_train, X_val, y_val, checkpoint=Config.CHECKPOINT, 
                        epochs=None, warmup_epochs=None, use_swa=None, use_progressive=None, use_augmentation=None):
    print("\n  [TRAIN] Advanced training suite (PyTorch) ...")
    
    # Use provided parameters or fallback to Config
    total_epochs = epochs if epochs is not None else Config.EPOCHS
    stage1_epochs = warmup_epochs if warmup_epochs is not None else Config.WARMUP_EPOCHS
    do_swa = use_swa if use_swa is not None else Config.USE_SWA
    do_prog = use_progressive if use_progressive is not None else Config.PROGRESSIVE_RES
    do_aug = use_augmentation if use_augmentation is not None else True # Default to True for main logic
    
    criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
    scaler = amp.GradScaler(enabled=(DEVICE.type != 'cpu'))
    
    # Stage 1: Warmup Head
    print("  Stage 1: Warming up head...")
    for param in model.backbone.parameters():
        param.requires_grad = False
        
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR_HEAD)
    
    # Data resolution for warmup
    res = Config.RES_SCHEDULE[0][1] if do_prog else Config.IMG_SIZE[0]
    train_transform = get_transforms(img_size=res, augment=do_aug)
    val_transform = get_transforms(img_size=res, augment=False)
    
    train_dataset = BrainTumorDataset(X_train, y_train, transform=train_transform)
    val_dataset = BrainTumorDataset(X_val, y_val, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # Warmup for WARMUP_EPOCHS
    for epoch in range(stage1_epochs):
        loss, acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
        vloss, vacc = validate_one_epoch(model, val_loader, criterion, DEVICE)
        print(f"    Epoch {epoch+1}/{stage1_epochs} - Loss: {loss:.4f} Acc: {acc:.4f} | Val Loss: {vloss:.4f} Val Acc: {vacc:.4f}")

    # Stage 2: Full Fine-tuning
    print("  Stage 2: Full fine-tuning...")
    for param in model.parameters():
        param.requires_grad = True

    # Create a NEW optimizer that includes ALL parameters (backbone + head)
    # with differential learning rates: lower LR for backbone, higher for head
    optimizer = optim.AdamW([
        {"params": model.backbone.parameters(), "lr": Config.LR_FT},
        {"params": model.pool_head.parameters(), "lr": Config.LR_FT * 5},
        {"params": model.classifier.parameters(), "lr": Config.LR_FT * 5},
    ], weight_decay=Config.WEIGHT_DECAY)
        
    early_stopping = EarlyStoppingPyTorch(patience=Config.PATIENCE, verbose=True, path=checkpoint)
    
    # Scheduler: OneCycleLR provides warmup, peak LR, and cosine decay
    # total_steps should account for epochs and steps per epoch
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[Config.LR_FT, Config.LR_FT * 5, Config.LR_FT * 5],
        epochs=total_epochs, steps_per_epoch=steps_per_epoch,
        pct_start=0.1, div_factor=10.0, final_div_factor=100.0
    )
    
    # Handle Progressive Resolution and SWA
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    if do_swa:
        from torch.optim.swa_utils import AveragedModel, SWALR
        swa_model = AveragedModel(model)
        swa_scheduler = SWALR(optimizer, swa_lr=Config.LR_FT)
        
    # Calculate proportional SWA start
    swa_start = int(0.75 * total_epochs) if total_epochs != Config.EPOCHS else Config.SWA_START_EPOCH

    curr_res = res
    for epoch in range(total_epochs):
        # Update resolution if needed
        if do_prog:
            for start_ep, new_res in Config.RES_SCHEDULE:
                if epoch == start_ep and new_res != curr_res:
                    print(f"    [RES] Updating resolution to {new_res}x{new_res}")
                    curr_res = new_res
                    train_loader.dataset.transform = get_transforms(img_size=curr_res, augment=do_aug)
                    val_loader.dataset.transform = get_transforms(img_size=curr_res, augment=False)

        # Train one epoch
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs}")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            with amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type != 'cpu')):
                outputs = model(images)
                # Label smoothing is already in criterion
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # Step the scheduler per batch
            scheduler.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix(lr=optimizer.param_groups[0]['lr'], loss=loss.item(), acc=100.*correct/total)
            
        tr_loss = running_loss / total
        tr_acc = correct / total
        
        # Validate
        vl_loss, vl_acc = validate_one_epoch(model, val_loader, criterion, DEVICE)
        
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)
        
        if do_swa and epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        
        early_stopping(vl_loss, model)
        if early_stopping.early_stop:
            print("  [INFO] Early stopping triggered.")
            break
            
    # Restore best weights
    model.load_state_dict(torch.load(checkpoint))
    print(f"  [INFO] Restored best weights from {checkpoint}")

    # Finalize SWA
    if do_swa:
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
        torch.save(swa_model.state_dict(), checkpoint.replace(".pth", "_swa.pth"))
        return history, swa_model
        
    return history, model

# ─────────────────────────────────────────────────────────────────────────────
# 6. Performance & Statistical Suite
# ─────────────────────────────────────────────────────────────────────────────
class TemperatureScaler(nn.Module):
    """
    Temperature Scaling for model calibration.
    Optimizes a single scalar T to minimize Negative Log Likelihood (NLL) on validation data.

    Acts as a transparent proxy for the wrapped model: any attribute not defined
    on TemperatureScaler itself (e.g. .backbone, .classifier) is automatically
    delegated to self.model via __getattr__.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)  # Initialize slightly > 1

    def __getattr__(self, name):
        # nn.Module.__getattr__ is only called when normal lookup fails, so
        # self.model and self.temperature are always found via the usual path.
        # We chain to PyTorch's own __getattr__ first (handles parameters /
        # buffers / submodules), then fall through to the wrapped model.
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.model, name)

    def _safe_temperature(self):
        """Returns temperature clamped to a safe positive range.
        clamp is preferred over abs: abs has a non-differentiable kink at zero
        that can destabilise LBFGS near the boundary."""
        return self.temperature.clamp(min=1e-4)

    def forward(self, x):
        """Forward pass takes images (x) and returns temperature-scaled logits."""
        logits = self.model(x)
        return logits / self._safe_temperature()

    def set_temperature(self, valid_loader, device):
        self.to(device) # Ensure the scaler parameters are on the same device as data
        self.model.eval()
        logits_list = []
        labels_list = []
        print("  [CALIBRATION] Extracting validation logits for temperature scaling...")
        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.to(device)
                logits = self.model(images)
                logits_list.append(logits)
                labels_list.append(labels)
            logits = torch.cat(logits_list)
            labels = torch.cat(labels_list)

        # Optimization: Minimize Cross Entropy (NLL) with respect to T
        nll_criterion = nn.CrossEntropyLoss()
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=100)

        def eval_loss():
            optimizer.zero_grad()
            # Use _safe_temperature() here too so the optimisation never
            # crosses into negative territory and inverts class rankings.
            loss = nll_criterion(logits / self._safe_temperature(), labels)
            loss.backward()
            return loss

        optimizer.step(eval_loss)
        t_val = self._safe_temperature().item()
        print(f"  [CALIBRATION] Optimization complete. Optimal T = {t_val:.4f}")
        return t_val

def calculate_brier_score(y_true_oh, y_prob):
    return np.mean(np.sum((y_prob - y_true_oh)**2, axis=1))

def mcnemar_test_holm(
    y_true: np.ndarray,
    proposed_pred: np.ndarray,
    baseline_preds: dict[str, np.ndarray],
    alpha: float = 0.05,
) -> list[dict[str, Any]]:
    """
    Paired McNemar tests vs each baseline, with Holm-Bonferroni family-wise error
    correction across the set of baselines per dataset.

    Uses the EXACT McNemar binomial when (n01+n10) < 25 (small-sample regime), and
    the continuity-corrected chi-square approximation otherwise. Statsmodels picks
    the right branch internally when given exact=True/False.

    Returns a list of result dicts (one per baseline), in the same order as
    `baseline_preds`. Each dict contains: baseline name, n01, n10, test_kind,
    raw p, Holm-adjusted p, reject_h0_after_correction.
    """
  
    y_true = np.asarray(y_true)
    proposed_pred = np.asarray(proposed_pred)
    c_p = (proposed_pred == y_true)

    names: list[str] = []
    raw_pvals: list[float] = []
    rows: list[dict[str, Any]] = []

    for name, p_b in baseline_preds.items():
        p_b = np.asarray(p_b)
        c_b = (p_b == y_true)
        n01 = int(np.sum(c_p & ~c_b))   # proposed right, baseline wrong
        n10 = int(np.sum(~c_p & c_b))   # proposed wrong, baseline right
        # mcnemar() expects a 2x2 contingency table; only the off-diagonals matter.
        table = [[0, n01], [n10, 0]]
        use_exact = (n01 + n10) < 25
        res = smm_mcnemar(table, exact=use_exact, correction=True)

        names.append(name)
        raw_pvals.append(float(res.pvalue))
        rows.append({
            "baseline": name,
            "n01_proposed_only_correct": n01,
            "n10_baseline_only_correct": n10,
            "test_kind": "exact" if use_exact else "chi2_continuity",
            "p_raw": float(res.pvalue),
        })

    # Holm-Bonferroni controls family-wise error across all 7 baseline comparisons.
    # Statsmodels' multipletests with method='holm' is the standard implementation.
    if len(raw_pvals) > 0:
        reject, p_adj, _, _ = multipletests(raw_pvals, alpha=alpha, method="holm")
        for i, row in enumerate(rows):
            row["p_holm"] = float(p_adj[i])
            row["reject_h0_after_correction"] = bool(reject[i])

    # Pretty-print the result table for the log.
    print(f"\n  [STAT] McNemar vs baselines (Holm-Bonferroni, alpha={alpha})")
    print(f"  {'Baseline':<18} {'n01':>5} {'n10':>5}  {'kind':<18} "
          f"{'raw p':>11} {'Holm p':>11}  sig")
    print("  " + "-" * 78)
    for r in rows:
        sig_str = "***" if r.get("reject_h0_after_correction") else " "
        print(f"  {r['baseline']:<18} {r['n01_proposed_only_correct']:>5} "
              f"{r['n10_baseline_only_correct']:>5}  {r['test_kind']:<18} "
              f"{r['p_raw']:>11.4g} {r['p_holm']:>11.4g}  {sig_str}")
    print("  " + "-" * 78)
    return rows

def compute_multiclass_specificity(y_true, y_pred, num_classes=Config.NUM_CLASSES):
    """
    Compute macro-averaging specificity for multiclass classification.
    """
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    spec = []
    for i in range(num_classes):
        tp = cm[i,i]
        fp = cm[:,i].sum() - tp
        fn = cm[i,:].sum() - tp
        tn = cm.sum() - tp - fp - fn
        s = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        spec.append(s)
    return np.mean(spec) # Standard macro averaging

def evaluate_model_pytorch(model, loader, device, label="Model"):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images) # If calibrated, this already applies T
            probs = F.softmax(outputs, dim=1)
            
            y_true.extend(labels.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())
            y_pred.extend(torch.argmax(probs, dim=1).cpu().numpy())
            
    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)
    y_true_oh = np.eye(Config.NUM_CLASSES)[y_true]
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    auc = roc_auc_score(y_true_oh, y_prob, multi_class="ovr", average="macro")
    kappa = cohen_kappa_score(y_true, y_pred)
    brier = calculate_brier_score(y_true_oh, y_prob)
    spec = compute_multiclass_specificity(y_true, y_pred)
    
    print(f"\n  [{label}] Performance:")
    print(f"    - Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc*100:.2f}%")
    print(f"    - Kappa: {kappa:.4f} | Brier: {brier:.4f} | Specificity: {spec*100:.2f}%")
    
    return y_pred, y_prob, confusion_matrix(y_true, y_pred), {
        "acc": acc, "prec": prec, "rec": rec, "f1": f1, 
        "auc": auc, "kappa": kappa, "brier": brier, "spec": spec
    }

def _unwrap_to_backbone(model):
    """
    Recursively unwrap any number of model wrappers until we reach a module
    that has a 'backbone' attribute (i.e. BrainTumorModel).
    """
    seen_ids = set()
    while not hasattr(model, 'backbone'):
        mid = id(model)
        if mid in seen_ids:
            break
        seen_ids.add(mid)
        if hasattr(model, 'module'):      # AveragedModel (torch.optim.swa_utils)
            model = model.module
        elif hasattr(model, 'model'):     # TemperatureScaler
            model = model.model
        else:
            break
    return model

def compute_computational_complexity(model, img_size=(224, 224), device=DEVICE):
    """Calculates Params (M), FLOPs (G), and Inference Time (ms)."""
    model.eval()
    dummy_input = torch.randn(1, 3, *img_size).to(device)
    
    # 1. Parameter Count
    params = sum(p.numel() for p in model.parameters()) / 1e6
    
    # 2. FLOPs
    flops = 0.0
    if thop:
        try:
            # Unwrap wrappers (SWA, TemperatureScaling) for thop to "see" the layers
            target_model = _unwrap_to_backbone(model)
            # Ensure input is same type as model (handles FP16/AMP edge cases)
            p_dtype = next(target_model.parameters()).dtype
            flops, _ = thop.profile(target_model, inputs=(dummy_input.to(p_dtype),), verbose=False)
            flops = flops / 1e9
        except Exception as e:
            # Log specific error to stderr (captured by Logger)
            print(f"  [DEBUG] thop.profile failed: {e}", file=sys.stderr)
            # flops remains 0.0, which triggers the fallback below
            
    if flops == 0.0:
        # Fallback Estimation Grid (Standard GFLOPs at 224x224)
        estimates = {
            "effnetv2_b0": 0.72, "resnet50": 4.12, "vgg16": 15.5, 
            "vgg19": 19.6, "densenet201": 4.3, "mobilenetv2": 0.32, 
            "convnext_tiny": 4.5, "swin_tiny": 4.5
        }
        m_str = str(_unwrap_to_backbone(model)).lower()
        for key, val in estimates.items():
            if key in m_str:
                flops = val
                break
            
    # 3. Inference Time
    for _ in range(10): _ = model(dummy_input) # Warmup
        
    if device.type == 'cpu':
        start_time = time.time()
        for _ in range(100): _ = model(dummy_input)
        infer_time = (time.time() - start_time) / 100 * 1000
    else:
        starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        times = []
        with torch.no_grad():
            for _ in range(100):
                starter.record()
                _ = model(dummy_input)
                ender.record()
                torch.cuda.synchronize()
                times.append(starter.elapsed_time(ender))
        infer_time = np.mean(times)
        
    return {"params": params, "flops": flops, "infer_time": infer_time}

def cross_validate_model_pytorch(X, y):
    print(f"\n  [STAT] {Config.N_FOLDS}-Fold CV (PyTorch) ...")
    skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=SEED)
    fold_accs = []
    
    for f, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n  --- CV Fold {f}/{Config.N_FOLDS} ---")
        model = BrainTumorModel().to(DEVICE)
        
        # Unique checkpoint per fold to avoid clobbering final model
        fold_ckpt = os.path.join(Config.OUTPUT_DIR, f"cv_fold_{f}_best_pytorch.pth")
        
        # Call the unified training suite with condensed epoch counts
        _, model = train_model_pytorch(
            model, X[tr_idx], y[tr_idx], X[va_idx], y[va_idx], 
            checkpoint=fold_ckpt,
            epochs=Config.CV_EPOCHS, 
            warmup_epochs=Config.CV_WARMUP_EPOCHS
        )
        
        # Evaluate best weights (train_model_pytorch ensures model has best weights)
        val_ds = BrainTumorDataset(X[va_idx], y[va_idx], transform=get_transforms(augment=False))
        val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False)
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        
        _, vacc = validate_one_epoch(model, val_loader, criterion, DEVICE)
        fold_accs.append(vacc)
        print(f"    Fold {f} Accuracy: {vacc*100:.2f}%")
        
    print(f"\n  [RESULT] CV Summary: {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%")
    return fold_accs

def plot_calibration_comparison_pytorch(y_true, y_prob_before, y_prob_after, label="Model"):
    """
    Side-by-side reliability diagrams showing before vs after calibration.
    """
    def get_stats(yt, yp):
        conf = yp.max(axis=1)
        p = yp.argmax(axis=1)
        acc_v = (p == yt).astype(float)
        ece = 0.0
        edges = np.linspace(0, 1, Config.ECE_BINS + 1)
        b_conf, b_acc = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            mask = (conf > lo) & (conf <= hi)
            if mask.any():
                acc = acc_v[mask].mean()
                cf = conf[mask].mean()
                b_acc.append(acc)
                b_conf.append(cf)
                ece += (mask.sum()/len(yt)) * abs(acc - cf)
        return b_conf, b_acc, ece

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    for i, (yp, sublbl) in enumerate([(y_prob_before, "Before Calibration"), (y_prob_after, "After Calibration")]):
        b_conf, b_acc, ece = get_stats(y_true, yp)
        ax = axes[i]
        ax.bar(b_conf, b_acc, width=0.8/Config.ECE_BINS, alpha=0.7, color="steelblue", label="Accuracy")
        ax.bar(b_conf, [bc-ba for bc,ba in zip(b_conf, b_acc)], bottom=b_acc, width=0.8/Config.ECE_BINS, alpha=0.3, color="crimson", label="Gap")
        ax.plot([0,1],[0,1],"k--")
        ax.set_xlim(0,1)
        ax.set_ylim(0,1)
        ax.set_title(f"{sublbl}\nECE={ece:.4f}")
        ax.legend()
        
    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, f"calibration_comparison_{label.replace(' ','_')}.pdf"), bbox_inches='tight')
    plt.close()
    print(f"  [PLOT] Calibration comparison saved to outputs/")

def compute_ece_and_reliability_diagram(y_true, y_prob, label="Model"):
    conf = y_prob.max(axis=1)
    p = y_prob.argmax(axis=1)
    acc_v = (p == y_true).astype(float)
    ece = 0.0
    edges = np.linspace(0, 1, Config.ECE_BINS + 1)
    b_conf, b_acc = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            acc = acc_v[mask].mean()
            cf = conf[mask].mean()
            b_acc.append(acc)
            b_conf.append(cf)
            ece += (mask.sum()/len(y_true)) * abs(acc - cf)
            
    fig, ax = plt.subplots(figsize=(5,5))
    ax.bar(b_conf, b_acc, width=0.8/Config.ECE_BINS, alpha=0.7, color="steelblue", label="Accuracy")
    ax.bar(b_conf, [bc-ba for bc,ba in zip(b_conf, b_acc)], bottom=b_acc, width=0.8/Config.ECE_BINS, alpha=0.3, color="crimson", label="Gap")
    ax.plot([0,1],[0,1],"k--")
    ax.set_xlim(0,1)
    ax.set_ylim(0,1)
    ax.set_title(f"Reliability: {label}\nECE={ece:.4f}")
    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, f"calibration_{label.replace(' ','_')}.pdf"), bbox_inches='tight')
    plt.close()
    return ece

def compute_random_iou_baseline(top_k_pct: float = 0.10) -> float:
    """
    Analytic expected IoU between TWO INDEPENDENT random binary masks, each selecting
    the top `top_k_pct` of N pixels.

        E[|A ∩ B|] / E[|A ∪ B|]  ≈  k² / (2k - k²)  where k = top_k_pct

    For k = 0.10 → 0.01 / 0.19 ≈ 0.0526.

    The manuscript currently states "≈ 0.10" as the random-mask baseline, which is
    actually the per-mask top-k fraction, NOT the expected IoU of two such masks.
    Use this value in Section 4.7 instead.
    """
    k = top_k_pct
    intersection = k * k
    union = 2 * k - k * k
    return intersection / union


def bootstrap_confidence_intervals(y_true, y_pred, y_prob, B=Config.BOOTSTRAP_B):
    n = len(y_true)
    rng = np.random.default_rng(SEED)
    y_oh = np.eye(Config.NUM_CLASSES)[y_true]

    def _m(yt, yp, ypr, yoh):
        a = accuracy_score(yt, yp)
        p = precision_score(yt, yp, average="weighted", zero_division=0)
        r = recall_score(yt, yp, average="weighted", zero_division=0)
        f = f1_score(yt, yp, average="weighted", zero_division=0)
        try:
            au = roc_auc_score(yoh, ypr, multi_class="ovr", average="macro")
        except:
            au = 0.5
        k = cohen_kappa_score(yt, yp)
        b = calculate_brier_score(yoh, ypr)
        s = compute_multiclass_specificity(yt, yp)
        return a, p, r, f, au, k, b, s

    boot = np.zeros((B, 8))
    for b in range(B):
        idx = rng.integers(0, n, n)
        boot[b] = _m(y_true[idx], y_pred[idx], y_prob[idx], y_oh[idx])

    point = _m(y_true, y_pred, y_prob, y_oh)
    lbls = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC", "Kappa", "Brier Score", "Specificity"]
    
    print(f"\n  [STAT] 95% Bootstrap CIs (B={B:,})")
    print("-" * 55)
    print(f"{'Metric':<20} | {'Point Est.':<12} | {'95% CI':<15}")
    print("-" * 55)
    for i, lbl in enumerate(lbls):
        lo, hi = np.percentile(boot[:, i], [2.5, 97.5])
        if lbl in ["Kappa", "Brier Score"]:
            print(f"  {lbl:<18} | {point[i]:10.4f}   | [{lo:.4f}, {hi:.4f}]")
        else:
            print(f"  {lbl:<18} | {point[i]*100:8.2f}%   | [{lo*100:.2f}%, {hi*100:.2f}%]")
    print("-" * 55)

def mcnemar_test(y_true, proposed_pred, baseline_preds):
    """
    This applies Holm-Bonferroni correction across the family of baseline
    comparisons (Comment 7 of supervisor review) and uses exact-binomial McNemar
    when (n01+n10) < 25 instead of always using the chi-square approximation.

    Returns a list of dicts with raw p, Holm-adjusted p, and significance flag —
    so callers can persist the results to the experiment_summary.json.
    """
    return mcnemar_test_holm(
        np.asarray(y_true),
        np.asarray(proposed_pred),
        baseline_preds,
        alpha=0.05,
    )
def bootstrap_confidence_intervals_returning(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    num_classes: int,
    B: int = 2000,
    seed: int = 42,
    calculate_brier_score_fn=None,
    compute_multiclass_specificity_fn=None,
) -> dict[str, dict[str, float]]:
    """
    Same as the notebook's bootstrap_confidence_intervals, but RETURNS the result so
    that we can serialise it to JSON.

    The two helper functions (Brier and macro-specificity) are passed in by the caller
    to avoid duplicating their definitions here.

    Returns:
        {
            "Accuracy":   {"point": 0.9602, "ci_low": 0.9388, "ci_high": 0.9817},
            "Precision":  {...},
            ...
        }
    """
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, cohen_kappa_score,
    )

    if calculate_brier_score_fn is None or compute_multiclass_specificity_fn is None:
        raise ValueError(
            "Pass calculate_brier_score_fn= and compute_multiclass_specificity_fn= "
            "from the notebook (these live in the main code, not here)."
        )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    rng = np.random.default_rng(seed)
    y_oh = np.eye(num_classes)[y_true]

    def _m(yt, yp, ypr, yoh):
        a = accuracy_score(yt, yp)
        p = precision_score(yt, yp, average="weighted", zero_division=0)
        r = recall_score(yt, yp, average="weighted", zero_division=0)
        f = f1_score(yt, yp, average="weighted", zero_division=0)
        try:
            au = roc_auc_score(yoh, ypr, multi_class="ovr", average="macro")
        except Exception:
            au = 0.5
        k = cohen_kappa_score(yt, yp)
        b = calculate_brier_score_fn(yoh, ypr)
        s = compute_multiclass_specificity_fn(yt, yp)
        return a, p, r, f, au, k, b, s

    boot = np.zeros((B, 8))
    for b in range(B):
        idx = rng.integers(0, n, n)
        boot[b] = _m(y_true[idx], y_pred[idx], y_prob[idx], y_oh[idx])
    point = _m(y_true, y_pred, y_prob, y_oh)

    labels = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC",
              "Kappa", "Brier", "Specificity"]
    out: dict[str, dict[str, float]] = {}
    for i, lbl in enumerate(labels):
        lo, hi = np.percentile(boot[:, i], [2.5, 97.5])
        out[lbl] = {
            "point": float(point[i]),
            "ci_low": float(lo),
            "ci_high": float(hi),
        }
    return out

def save_experiment_summary(out_dir: str, payload: dict[str, Any]) -> str:
    """
    Atomic-write the experiment summary as pretty-printed JSON. This file is the
    SINGLE source of truth for the manuscript: every number in every table in
    Section 4 should be traceable to a key in this JSON.

    Atomic write protects against partial files if the kernel dies during writing.
    """
    path = os.path.join(out_dir, "experiment_summary.json")
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2, default=str)
    os.replace(tmp_path, path)
    print(f"\n  [SAVED] Persistent summary → {path}")
    return path

def selective_prediction_analysis(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    coverage_grid: list[float] | None = None,
) -> dict[str, Any]:
    """
    Selective-prediction / risk-coverage analysis: if we abstain on the lowest-confidence
    predictions, what does accuracy on the retained set look like?

    This directly addresses ("clinical relevance / triage / abstention").
    Output is consumable as a risk-coverage curve.

    Args:
        coverage_grid: list of coverage fractions ∈ (0, 1]. Default 0.50…1.00 step 0.05.

    Returns:
        {
            "coverage": [0.50, 0.55, ..., 1.00],
            "accuracy_at_coverage": [0.9987, 0.9971, ..., 0.9602],
            "confidence_threshold": [tau_0.50, tau_0.55, ..., 0.0],
        }
    """
    if coverage_grid is None:
        coverage_grid = [round(0.50 + 0.05 * i, 2) for i in range(11)]

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    conf = y_prob.max(axis=1)  # max-softmax confidence
    correct = (y_pred == y_true).astype(np.float64)
    n = len(y_true)

    order = np.argsort(-conf)  # high → low confidence
    sorted_correct = correct[order]
    sorted_conf = conf[order]

    out_cov: list[float] = []
    out_acc: list[float] = []
    out_tau: list[float] = []
    for c in coverage_grid:
        k = max(1, int(round(c * n)))
        acc_at_c = float(sorted_correct[:k].mean())
        tau_at_c = float(sorted_conf[k - 1])  # the lowest-kept confidence
        out_cov.append(float(c))
        out_acc.append(acc_at_c)
        out_tau.append(tau_at_c)

    return {
        "coverage": out_cov,
        "accuracy_at_coverage": out_acc,
        "confidence_threshold": out_tau,
    }

def build_summary_payload(
    *,
    dataset_key: str,
    dataset_name: str,
    seed: int,
    n_train: int, n_val: int, n_cal: int, n_test: int,
    per_class_support_test: dict[int, int],
    label_encoder=None,
    cv_acc_mean: float, cv_acc_std: float,
    cv_fold_accs: list[float],
    test_metrics_calibrated: dict[str, float],
    test_metrics_uncalibrated: dict[str, float],
    bootstrap_cis_calibrated: dict[str, dict[str, float]],
    bootstrap_cis_baselines: dict[str, dict[str, dict[str, float]]] | None,
    ece_uncalibrated: float,
    ece_calibrated: float,
    temperature: float,
    params_M: float, flops_G: float, inference_ms: float,
    ablation_rows: list[dict[str, Any]],
    mcnemar_rows: list[dict[str, Any]],
    xai_iou_per_class: dict[str, float],
    xai_iou_random_baseline: float,
    deletion_insertion_auc: dict[str, dict[str, Any]],
    selective_prediction: dict[str, Any],
    duplicate_detection_phash: dict[str, Any] | None,
    duplicate_detection_deep: dict[str, Any] | None,
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """
    Assemble the complete experiment_summary.json payload. Every numerical claim in
    the manuscript should be traceable to a key in this dict.
    """
    payload = {
        "dataset_key": dataset_key,
        "dataset_name": dataset_name,
        "seed": seed,
        "splits": {
            "n_train": int(n_train),
            "n_val": int(n_val),
            "n_cal": int(n_cal),
            "n_test": int(n_test),
            # Human-readable keys: map integer-encoded labels back to class names
            # using the LabelEncoder if provided. Falls back to "class_N" strings.
            "per_class_support_test": (
                {str(label_encoder.inverse_transform([int(k)])[0]): int(v)
                 for k, v in per_class_support_test.items()}
                if label_encoder is not None
                else {f"class_{int(k)}": int(v) for k, v in per_class_support_test.items()}
            ),
        },
        "cross_validation": {
            "mean_acc": float(cv_acc_mean),
            "std_acc": float(cv_acc_std),
            "fold_accs": [float(a) for a in cv_fold_accs],
        },
        "test_calibrated": {k: float(v) for k, v in test_metrics_calibrated.items()},
        "test_uncalibrated": {k: float(v) for k, v in test_metrics_uncalibrated.items()},
        "bootstrap_ci_calibrated": bootstrap_cis_calibrated,
        "bootstrap_ci_baselines": bootstrap_cis_baselines or {},
        "calibration": {
            "ece_uncalibrated": float(ece_uncalibrated),
            "ece_calibrated": float(ece_calibrated),
            "ece_reduction_pct": float(100.0 * (1.0 - ece_calibrated / max(ece_uncalibrated, 1e-12))),
            "optimal_T": float(temperature),
        },
        "complexity": {
            "params_M": float(params_M),
            "flops_G": float(flops_G),
            "inference_ms": float(inference_ms),
        },
        "ablation": ablation_rows,
        "mcnemar_holm": mcnemar_rows,
        "xai": {
            # iou_per_class values are nested dicts (per-method-pair IoUs), NOT scalars.
            # Pass them through as-is rather than coercing with float(). Robust to both
            # nested-dict and scalar forms in case the XAI capture changes.
            "iou_per_class": {
                k: (v if isinstance(v, dict) else float(v))
                for k, v in xai_iou_per_class.items()
            },
            "iou_random_baseline_analytic": float(xai_iou_random_baseline),
            "deletion_insertion_auc": deletion_insertion_auc,
        },
        "selective_prediction": selective_prediction,
        "duplicate_detection_phash": duplicate_detection_phash,
        "duplicate_detection_deep": duplicate_detection_deep,
    }
    if extra:
        payload["extra"] = extra
    return payload
# ─────────────────────────────────────────────────────────────────────────────
# 7. XAI Suite (PyTorch Hooks)
# ─────────────────────────────────────────────────────────────────────────────
class GradCAMPlusPlusPyTorch:
    """
    True Grad-CAM++ (Chattopadhay et al., 2018).
    Uses pixel-wise second-order (squared) gradient weights — Issue 4 fix.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activations)
        target_layer.register_full_backward_hook(self.save_gradients)

    def save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def save_activations(self, module, input, output):
        self.activations = output

    def __call__(self, x, class_idx):
        self.model.eval()
        logits = self.model(x)
        self.model.zero_grad()
        loss = logits[0, class_idx]
        loss.backward()

        with torch.no_grad():
            grads = self.gradients   # (1, C, H, W)
            acts  = self.activations  # (1, C, H, W)

            # True Grad-CAM++ alpha weights
            grads_sq = grads.pow(2)  # (∂y/∂A)²
            grads_cu = grads.pow(3)  # (∂y/∂A)³

            # Global sum: Σ_{a,b} A^k_{ab} · (∂²y/∂A²) ≈ acts * grads³
            global_sum = (acts * grads_cu).sum(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
            alpha_den  = 2 * grads_sq + global_sum
            alpha      = grads_sq / (alpha_den + 1e-7)        # (1, C, H, W)

            
            weights = (alpha * grads).sum(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)

            cam = (weights * acts).sum(dim=1, keepdim=True)   # (1, 1, H, W)
            cam = F.relu(cam)
            cam = F.interpolate(cam, size=Config.IMG_SIZE, mode='bilinear', align_corners=False)
            cam = cam.squeeze().cpu().numpy()

            if cam.max() > 0:
                cam = cam / cam.max()
            return cam

def compute_integrated_gradients_pytorch(model, img_tensor, class_idx, steps=Config.DI_STEPS):
    model.eval()
    baseline = torch.zeros_like(img_tensor).to(DEVICE)
    alphas = torch.linspace(0, 1, steps + 1).to(DEVICE)
    
    grads_list = []
    for alpha in alphas:
        interpolated = baseline + alpha * (img_tensor - baseline)
        interpolated.requires_grad = True
        
        logits = model(interpolated)
        loss = logits[0, class_idx]
        model.zero_grad()
        loss.backward()
        
        grads_list.append(interpolated.grad.detach())
        
    avg_grads = torch.stack(grads_list).mean(dim=0)
    ig = (img_tensor - baseline) * avg_grads
    saliency = ig.abs().sum(dim=1).squeeze().cpu().numpy()
    
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    return saliency

def explain_with_shap_pytorch(model, X_test, y_test_int):
    print("\n  [XAI] SHAP (GradientExplainer for PyTorch)...")
    model.eval()
    
    # Background for shap
    bg_idx = np.random.choice(len(X_test), min(Config.SHAP_BG_SIZE, len(X_test)), replace=False)
    # Correct transform for background
    tfm = get_transforms(augment=False)
    background = torch.stack([tfm(X_test[i]) for i in bg_idx]).to(DEVICE)
    
    explainer = shap.GradientExplainer(model, background)
    
    sample_imgs = []
    sample_classes = []
    for ci in range(Config.NUM_CLASSES):
        idx = np.where(y_test_int == ci)[0]
        if len(idx) > 0:
            sample_imgs.append(tfm(X_test[idx[0]]))
            sample_classes.append(ci)
            
    sample_imgs = torch.stack(sample_imgs).to(DEVICE)
    shap_values = explainer.shap_values(sample_imgs)
    
    if isinstance(shap_values, list):
        # Older SHAP versions return a list of arrays (one per class)
        return {cls: shap_values[cls][i] for i, cls in enumerate(sample_classes)}
    else:
        # Newer SHAP versions return a single array with class dimension at the end: (N, C, H, W, K)
        # We extract the specific class for each sample
        return {cls: shap_values[i][..., cls] for i, cls in enumerate(sample_classes)}

def compute_deletion_insertion_pytorch(model, img_tensor, attr_map, class_idx, temperature=1.0):
    """
    Computes Deletion (lower is better) and Insertion (higher is better) curves.
    """
    model.eval()
    img_tensor = img_tensor.to(DEVICE) # (1, 3, H, W)
    _, _, H, W = img_tensor.shape
    flat_size = H * W
    
    # Sort pixels by importance
    attr_flat = attr_map.ravel()
    order = np.argsort(attr_flat)[::-1].copy()
    
    # Baseline for Insertion: Blurred version of the image
    img_np = img_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    blurred = cv2.GaussianBlur(img_np, (51, 51), 10)
    blurred_tensor = torch.from_numpy(blurred).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
    
    # Grey value in normalized space: (0.5 - mean) / std
    # Using ImageNet constants: R:(0.5-0.485)/0.229=0.0655, G:(0.5-0.456)/0.224=0.1964, B:(0.5-0.406)/0.225=0.4178
    grey_val = torch.tensor([0.0655, 0.1964, 0.4178]).view(1, 3, 1, 1).to(DEVICE)
    
    d_curve, i_curve = [], []
    steps = Config.DI_STEPS
    step_size = flat_size // steps
    
    with torch.no_grad():
        for s in range(steps + 1):
            n_pixels = min(s * step_size, flat_size)
            idx = order[:n_pixels]
            
            # Deletion: Start with original, replace important with mean/grey
            d_img = img_tensor.clone()
            # Reshape to (1, 3, -1) to apply indices easily
            d_img_flat = d_img.view(1, 3, -1)
            for c in range(3):
                d_img_flat[0, c, idx] = grey_val[0, c, 0, 0]
            
            # Insertion: Start with blurred, add important from original
            i_img = blurred_tensor.clone()
            i_img_flat = i_img.view(1, 3, -1)
            img_tensor_flat = img_tensor.view(1, 3, -1)
            for c in range(3):
                i_img_flat[0, c, idx] = img_tensor_flat[0, c, idx]
            
            # Predict with temperature scaling
            d_logits = model(d_img)
            i_logits = model(i_img)
            if temperature != 1.0:
                d_logits = d_logits / temperature
                i_logits = i_logits / temperature
                
            d_prob = F.softmax(d_logits, dim=1)[0, class_idx].item()
            i_prob = F.softmax(i_logits, dim=1)[0, class_idx].item()
            
            d_curve.append(d_prob)
            i_curve.append(i_prob)
            
    return np.array(d_curve), np.array(i_curve)

def plot_deletion_insertion_pytorch(model, X_test, y_test_int, shap_results, temperature=1.0):
    """
    Deletion/Insertion curves averaged over K samples per class.
    Provides statistically robust perceptual XAI evaluation.
    """
    print("\n  [XAI] Generating Deletion/Insertion Curves (K-averaged)...")
    K = max(1, min(50, len(X_test) // Config.NUM_CLASSES))  
    tfm = get_transforms(augment=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ci in range(Config.NUM_CLASSES):
        class_idx = np.where(y_test_int == ci)[0]
        if not len(class_idx):
            continue

        sh = shap_results.get(ci, np.zeros((224, 224)))
        if sh.ndim == 3:
            sh = np.abs(sh).mean(axis=0)   # (H, W)
        sh = sh / (sh.max() + 1e-8)

        d_all, i_all = [], []
        for sample_idx in class_idx[:K]:
            img = X_test[sample_idx]
            img_tensor = tfm(img).unsqueeze(0).to(DEVICE)
            d, i = compute_deletion_insertion_pytorch(model, img_tensor, sh, ci)
            d_all.append(d)
            i_all.append(i)

        x_axis = np.linspace(0, 1, len(d_all[0]))
        # Per-sample AUCs (trapezoidal rule over [0,1])
        try:
            trapezoid_auc = np.trapezoid
        except AttributeError:
            trapezoid_auc = np.trapz
        d_aucs = [float(trapezoid_auc(d, dx=1.0/len(d))) for d in d_all]
        i_aucs = [float(trapezoid_auc(i, dx=1.0/len(i))) for i in i_all]
        # Bootstrap 95% CIs on the per-class mean AUC (Comment 8b)
        def _boot_ci(arr, B=2000, alpha=0.05):
            rng = np.random.default_rng(SEED + ci)
            a = np.asarray(arr)
            means = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(B)]
            lo, hi = np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])
            return float(lo), float(hi)
        d_lo, d_hi = _boot_ci(d_aucs)
        i_lo, i_hi = _boot_ci(i_aucs)
        _XAI_DEL_INS_AUC[Config.CLASS_LABELS[ci]] = {
            "deletion_auc_mean":  float(np.mean(d_aucs)),
            "deletion_auc_ci":    [d_lo, d_hi],
            "insertion_auc_mean": float(np.mean(i_aucs)),
            "insertion_auc_ci":   [i_lo, i_hi],
            "n_samples": len(d_all),
        }
        axes[0].plot(x_axis, np.mean(d_all, axis=0), label=Config.CLASS_LABELS[ci])
        axes[1].plot(x_axis, np.mean(i_all, axis=0), label=Config.CLASS_LABELS[ci])

    axes[0].set_title(f"Deletion – Mean over {K} samples (Lower AUC is Better)")
    axes[0].set_xlabel("Fraction of pixels removed")
    axes[0].set_ylabel("Confidence")
    axes[0].legend()

    axes[1].set_title(f"Insertion – Mean over {K} samples (Higher AUC is Better)")
    axes[1].set_xlabel("Fraction of pixels added")
    axes[1].set_ylabel("Confidence")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, "del_ins_pytorch.pdf"), bbox_inches='tight')
    plt.close()

# ─────────────────────────────────────────────────────────────────────────────
# 8. Ablation Study
# ─────────────────────────────────────────────────────────────────────────────
def ablation_study_pytorch(X_train, y_train, X_val, y_val, X_test, y_test, le):
    """
    Strategic 5-stage ablation study with two-stage training and McNemar testing.
    """
    print("\n" + "="*95)
    print("  STRATEGIC ABLATION STUDY: COMPONENT-WISE CONTRIBUTION (PyTorch)")
    print("="*95)
    
    # Configuration Grid (Cumulative)
    # (ID, Label, Use_Attention, Use_Aug, Use_SWA, Use_Prog)
    cfgs = [
        (1, "Baseline (EffNetV2-B0 + GAP)",         False, False, False, False),
        (2, "Stage 1: + Dual-Pooling Attention",    True,  False, False, False),
        (3, "Stage 2: + Extended Augmentation",     True,  True,  False, False),
        (4, "Stage 3: + Stochastic Weight Avg",     True,  True,  True,  False),
        (5, "Stage 4: + Progressive Res (Proposed)",True,  True,  True,  True),
    ]
    
    results = []
    all_preds = {}
    y_test_oh = np.eye(Config.NUM_CLASSES)[y_test]
    
    for eid, lbl, use_attn, use_aug, use_swa, use_prog in cfgs:
        print(f"\n  [EXP {eid}] {lbl} ...")
        
        # 1. Initialize Model
        model = BrainTumorModel(use_attention=use_attn).to(DEVICE)
        
        # 2. Setup Data Loaders (Resolution based on whether Progressive is enabled)
        res = Config.RES_SCHEDULE[0][1] if use_prog else Config.IMG_SIZE[0]
        train_ds = BrainTumorDataset(X_train, y_train, transform=get_transforms(img_size=res, augment=use_aug))
        val_ds = BrainTumorDataset(X_val, y_val, transform=get_transforms(img_size=res, augment=False))
        
        train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False)
        
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scaler = amp.GradScaler(enabled=(DEVICE.type != 'cpu'))
        
        # ─────────────────────────────────────────────────────────────────────
        # Unified Training (Ablation)
        # ─────────────────────────────────────────────────────────────────────
        fold_ckpt = os.path.join(Config.OUTPUT_DIR, f"ablation_{eid}_best_pytorch.pth")
        _, eval_model = train_model_pytorch(
            model, X_train, y_train, X_val, y_val, 
            checkpoint=fold_ckpt,
            epochs=Config.ABLATION_EPOCHS,
            warmup_epochs=Config.ABLATION_HEAD_EPOCHS,
            use_swa=use_swa,
            use_progressive=use_prog,
            use_augmentation=use_aug
        )
            
        # 3. Evaluate on Test Set
        # Ensure test set is correct resolution for the final model
        test_res = Config.IMG_SIZE[0]
        test_ds = BrainTumorDataset(X_test, y_test, transform=get_transforms(img_size=test_res, augment=False))
        test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False)
        
        y_p, y_prob, _, metrics = evaluate_model_pytorch(eval_model, test_loader, DEVICE, label=lbl)
        
        # Complexity
        # Note: We compute complexity for the base model architecture at this stage
        # The inference time might vary based on resolution
        comp_res = compute_computational_complexity(eval_model, img_size=(test_res, test_res))
        
        metrics["ID"] = eid
        metrics["Label"] = lbl
        metrics["Params"] = comp_res["params"]
        metrics["FLOPs"] = comp_res["flops"]
        metrics["Time"] = comp_res["infer_time"]
        
        results.append(metrics)
        all_preds[eid] = y_p
        
        # Cleanup
        del model
        if use_swa: del eval_model # eval_model is a reference to swa_model here
        torch.cuda.empty_cache()
        gc.collect()

    # 4. Reporting & McNemar Tests (Proposed vs Others)
    print("\n" + "-"*230)
    print(f"{'ID':<3} | {'Configuration':<35} | {'Acc%':<6} | {'P%':<5} | {'R%':<5} | {'S%':<5} | {'F1%':<5} | {'Kappa':<6} | {'AUC':<5} | {'Params':<6} | {'FLOPs':<5} | {'Time':<5} | {'McNemar':<15}")
    print("-" * 230)
    
    prop_id = 5
    prop_preds = all_preds[prop_id]

    # Build a dict of {non-proposed ablation label: predictions} so we can apply
    # ONE Holm-Bonferroni correction across the family of ablation comparisons.
    
    _ablation_baseline_preds = {
        f"abl_{r['ID']}_{r['Label'][:25]}": all_preds[r['ID']]
        for r in results if r['ID'] != prop_id
    }
    _ablation_mcnemar_rows = mcnemar_test_holm(
        np.asarray(y_test),
        np.asarray(prop_preds),
        _ablation_baseline_preds,
        alpha=0.05,
    )
    # Store on the module-level list so the JSON writer can pick it up.
    _MCNEMAR_ROWS_ABLATION.extend([
        {**row, "comparison_family": "ablation"} for row in _ablation_mcnemar_rows
    ])
    # Build a lookup: ablation_id -> Holm-adjusted p-value (for the printed table).
    _holm_p_by_abl_id = {
        int(row["baseline"].split("_")[1]): row["p_holm"]
        for row in _ablation_mcnemar_rows
    }

    for r in results:
        eid = r['ID']
        if eid != prop_id:
            p_holm = _holm_p_by_abl_id.get(eid, float("nan"))
            mc_str = f"p_holm={p_holm:.4g}"
        else:
            mc_str = "Proposed"
            
        line = (f"{eid:<3} | {r['Label']:<35} | {r['acc']*100:6.1f} | {r['prec']*100:5.1f} | "
                f"{r['rec']*100:5.1f} | {r['spec']*100:5.1f} | {r['f1']*100:5.1f} | "
                f"{r['kappa']:6.3f} | {r['auc']:5.3f} | {r['Params']:6.2f} | {r['FLOPs']:5.2f} | {r['Time']:5.1f} | {mc_str:<15}")
        print(line)
    print("-" * 195)

    # Persist ablation results to module-level list so the JSON writer can pick them up.
    # Each row is sanitised — only basic Python types so it serialises cleanly to JSON.
    _ABLATION_RESULTS.extend([
        {k: (v if isinstance(v, (str, int, float, bool, list, dict, type(None)))
              else float(v) if hasattr(v, "__float__") else str(v))
         for k, v in r.items()}
        for r in results
    ])

    return results

# ─────────────────────────────────────────────────────────────────────────────
# 9. Baseline Comparison Suite
# ─────────────────────────────────────────────────────────────────────────────
def run_baseline_comparison_pytorch(X_train, y_train, X_val, y_val, X_test, y_test, proposed_pred, proposed_prob):
    """
    Compares the Proposed model against 7 frozen baselines:
    ResNet50, VGG16, VGG19, DenseNet201, MobileNet, ConvNeXt, Swin.
    Returns the full results list. Comment 7: applies Holm-Bonferroni across
    baselines and stashes the corrected results in _MCNEMAR_ROWS_BASELINE.
    """
    _baseline_preds_by_name: dict = {}
    _baseline_probs_by_name: dict = {}
    print("\n" + "="*95)
    print("  COMPREHENSIVE BASELINE COMPARISON (Frozen Backbones vs. Proposed)")
    print("="*95)
    
    baselines = [
        ("ResNet50",    "resnet50.a1_in1k"),
        ("VGG16",       "vgg16.tv_in1k"),
        ("VGG19",       "vgg19.tv_in1k"),
        ("DenseNet201", "densenet201.tv_in1k"),
        ("MobileNetV2", "mobilenetv2_100.ra_in1k"),
        ("ConvNeXt-T",  "convnext_tiny.in12k_ft_in1k"),
        ("Swin-T",      "swin_tiny_patch4_window7_224.ms_in1k"),
    ]
    
    # 1. Capture Proposed Results (passed as arguments)
    y_test_oh = np.eye(Config.NUM_CLASSES)[y_test]
    
    # We re-evaluate or use passed props
    proposed_acc = accuracy_score(y_test, proposed_pred)
    proposed_prec = precision_score(y_test, proposed_pred, average="weighted", zero_division=0)
    proposed_rec = recall_score(y_test, proposed_pred, average="weighted", zero_division=0)
    proposed_f1 = f1_score(y_test, proposed_pred, average="weighted", zero_division=0)
    proposed_auc = roc_auc_score(y_test_oh, proposed_prob, multi_class="ovr", average="macro")
    proposed_kappa = cohen_kappa_score(y_test, proposed_pred)
    proposed_brier = calculate_brier_score(y_test_oh, proposed_prob)
    proposed_spec = compute_multiclass_specificity(y_test, proposed_pred)
    
    # 0. Get Proposed Complexity
    comp_prop = compute_computational_complexity(BrainTumorModel().to(DEVICE))
    
    results = [{
        "Model": "**EffNetV2 (Proposed)**",
        "Acc": proposed_acc,
        "Prec": proposed_prec,
        "Rec": proposed_rec,
        "F1": proposed_f1,
        "AUC": proposed_auc,
        "Kappa": proposed_kappa,
        "Brier": proposed_brier,
        "Spec": proposed_spec,
        "Params": comp_prop["params"],
        "FLOPs": comp_prop["flops"],
        "Time": comp_prop["infer_time"],
        "McNemar": "-",
        "Sig": "-"
    }]
    
    # 2. Iterate through Baselines
    tfm = get_transforms(augment=False) # No aug for baseline frozen training
    test_loader = DataLoader(BrainTumorDataset(X_test, y_test, transform=tfm), batch_size=Config.BATCH_SIZE, shuffle=False)
    
    for name, model_tag in baselines:
        print(f"  [BASELINE] Training {name} (Frozen Backbone)...")
        
        # Create model with frozen backbone
        model = timm.create_model(model_tag, pretrained=True, num_classes=Config.NUM_CLASSES).to(DEVICE)
        for param in model.parameters():
            param.requires_grad = False
            
        # Unfreeze head (using robust get_classifier or common attribute names)
        unfrozen = False
        if hasattr(model, 'get_classifier'):
            classifier = model.get_classifier()
            if isinstance(classifier, nn.Module):
                for param in classifier.parameters():
                    param.requires_grad = True
                unfrozen = True
        
        if not unfrozen:
            for attr in ['head', 'classifier', 'fc']:
                if hasattr(model, attr):
                    module = getattr(model, attr)
                    if isinstance(module, nn.Module):
                        for param in module.parameters():
                            param.requires_grad = True
                        unfrozen = True
                        break
        
        trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))
        if not trainable_params:
            raise ValueError(f"Optimizer failed: No trainable parameters found for model '{name}' ({model_tag}). "
                             "This usually means the classification head wasn't identified correctly.")

        optimizer = optim.Adam(trainable_params, lr=1e-3)
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scaler = amp.GradScaler(enabled=(DEVICE.type != 'cpu'))
        
        train_ds = BrainTumorDataset(X_train, y_train, transform=tfm)
        val_ds = BrainTumorDataset(X_val, y_val, transform=tfm)
        train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False)
        
        # Train for 8 epochs
        for epoch in range(Config.BASELINE_EPOCHS):
            train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
            
        # Evaluate
        y_p, y_pb, _, res = evaluate_model_pytorch(model, test_loader, DEVICE, label=name)
        
        # Stash the baseline predictions for the family-wise Holm-Bonferroni
        # correction that runs AFTER the loop. Comment 7 of supervisor review.
        _baseline_preds_by_name[name] = y_p.copy()
        _baseline_probs_by_name[name] = y_pb.copy()
        # Raw uncorrected p (for the printed table only — final p in JSON is Holm-adjusted)
        c_p = (proposed_pred == y_test)
        c_b = (y_p == y_test)
        n01 = int(np.sum(c_p & ~c_b))
        n10 = int(np.sum(~c_p & c_b))
        from statsmodels.stats.contingency_tables import mcnemar as _smm_mc
        _use_exact = (n01 + n10) < 25
        _res = _smm_mc([[0, n01], [n10, 0]], exact=_use_exact, correction=True)
        pval = float(_res.pvalue)

        # Complexity
        comp = compute_computational_complexity(model)
        
        results.append({
            "Model": name,
            "Acc": res["acc"],
            "Prec": res["prec"],
            "Rec": res["rec"],
            "F1": res["f1"],
            "AUC": res["auc"],
            "Kappa": res["kappa"],
            "Brier": res["brier"],
            "Spec": res["spec"],
            "Params": comp["params"],
            "FLOPs": comp["flops"],
            "Time": comp["infer_time"],
            "McNemar": f"p={pval:.4f}",
            "Sig": "YES" if pval < 0.05 else "no"
        })
        
        # Cleanup
        del model, train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()

    # Holm-Bonferroni correction across all baseline comparisons (Comment 7).
    if _baseline_preds_by_name:
        _baseline_mcnemar_rows = mcnemar_test_holm(
            np.asarray(y_test),
            np.asarray(proposed_pred),
            _baseline_preds_by_name,
            alpha=0.05,
        )
        _MCNEMAR_ROWS_BASELINE.extend([
            {**row, "comparison_family": "baselines"} for row in _baseline_mcnemar_rows
        ])
        # Build a lookup: baseline name -> Holm-adjusted p for the printed table.
        _holm_p_by_baseline = {row["baseline"]: row["p_holm"] for row in _baseline_mcnemar_rows}
        # Overwrite the McNemar column in the results list with Holm-adjusted p.
        for r in results:
            if r["Model"] in _holm_p_by_baseline:
                p_h = _holm_p_by_baseline[r["Model"]]
                r["McNemar"] = f"p_holm={p_h:.4g}"
                r["Sig"]     = "YES" if p_h < 0.05 else "no"
        # Per-baseline bootstrap CIs (Improvements #4): cheap, no retraining.
        # We compute them here from the already-collected predictions.
        _baseline_bootstrap_cis: dict = {}
        for name, y_p in _baseline_preds_by_name.items():
            y_pb_b = _baseline_probs_by_name[name]
            try:
                _baseline_bootstrap_cis[name] = bootstrap_confidence_intervals_returning(
                    y_test, y_p, y_pb_b,
                    num_classes=Config.NUM_CLASSES,
                    B=Config.BOOTSTRAP_B, seed=SEED,
                    calculate_brier_score_fn=calculate_brier_score,
                    compute_multiclass_specificity_fn=compute_multiclass_specificity,
                )
            except Exception as _e:
                print(f"  [WARN] baseline bootstrap failed for {name}: {_e}")
        _BASELINE_BOOTSTRAP_CIS.update(_baseline_bootstrap_cis)
        # Also persist raw predictions so future analyses don't need to retrain.
        _BASELINE_PREDICTIONS.update({
            name: {"y_pred": y_p.tolist(), "y_prob_shape": list(_baseline_probs_by_name[name].shape)}
            for name, y_p in _baseline_preds_by_name.items()
        })

    # Persist BASELINE results to module-level list so the JSON writer can pick them up.
    _BASELINE_RESULTS.extend([dict(r) for r in results])

    # 3. Final Summary Table
    print("\n" + "-"*230)
    head = f"{'Model':<25} | {'Acc%':<6} | {'P%':<5} | {'R%':<5} | {'S%':<5} | {'F1%':<5} | {'Kappa':<6} | {'AUC':<5} | {'Params':<6} | {'FLOPs':<5} | {'Time':<5} | {'McNemar':<10} | {'Sig':<4}"
    print(head)
    print("-" * 230)
    for r in results:
        line = (f"{r['Model']:<25} | {r['Acc']*100:6.1f} | {r['Prec']*100:5.1f} | {r['Rec']*100:5.1f} | "
                f"{r['Spec']*100:5.1f} | {r['F1']*100:5.1f} | {r['Kappa']:6.3f} | {r['AUC']:5.3f} | "
                f"{r['Params']:6.2f} | {r['FLOPs']:5.2f} | {r['Time']:5.1f} | {r['McNemar']:<10} | {r['Sig']:<4}")
        print(line)
    print("-" * 230)
    
    return results

# ─────────────────────────────────────────────────────────────────────────────
# 10. Plotting & Final Assembly
# ─────────────────────────────────────────────────────────────────────────────
def plot_learning_curves(history, label):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].plot(history['train_loss'], label='Train')
    ax[0].plot(history['val_loss'], label='Val')
    ax[0].set_title('Loss')
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend()
    ax[1].plot(history['train_acc'], label='Train')
    ax[1].plot(history['val_acc'], label='Val')
    ax[1].set_title('Accuracy'); ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, f"learning_{label}.pdf"), bbox_inches='tight')
    plt.close()

def plot_confusion_matrix_pytorch(cm, label):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=Config.CLASS_LABELS, yticklabels=Config.CLASS_LABELS)
    plt.title(f"Confusion Matrix: {label}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, f"cm_{label.replace(' ','_')}.pdf"), bbox_inches='tight')
    plt.close()

def plot_roc_curves_pytorch(y_true, y_prob, label):
    y_oh = np.eye(Config.NUM_CLASSES)[y_true]
    plt.figure(figsize=(10, 8))
    colors = ['blue', 'red', 'green', 'orange']
    for i in range(Config.NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_true == i, y_prob[:, i])
        auc_val = sk_auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i],
                 label=f"{Config.CLASS_LABELS[i]} (AUC={auc_val:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves: {label}"); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, f"roc_{label.replace(' ','_')}.pdf"), bbox_inches='tight')
    plt.close()

def compute_xai_agreement_pytorch(sh, gc, ig):
    """Quantifies top-10% IoU agreement between XAI maps."""
    def _threshold(m):
        th = np.percentile(m, 100 * (1 - Config.XAI_TOP_K_PCT))
        return (m >= th).astype(np.uint8)
    ms, mg, mi = _threshold(sh), _threshold(gc), _threshold(ig)
    def _iou(a, b):
        u = (a | b).sum()
        return float((a & b).sum()) / u if u > 0 else 0.0
    sg, si, gi = _iou(ms, mg), _iou(ms, mi), _iou(mg, mi)
    return {"SG": sg, "SI": si, "GI": gi, "mean": (sg + si + gi) / 3.0}

def plot_xai_ensemble_pytorch(model, X_test, y_test_int, shap_results):
    print("\n  [XAI] Generating Ensemble visualizations (PyTorch)...")
    model.eval()
    # Unwrap all wrappers (TemperatureScaler, AveragedModel, etc.) to reach
    # the raw BrainTumorModel so that:
    #   (a) hook registration lands on the correct Conv2d layers, and
    #   (b) gradient magnitudes are not distorted by temperature scaling or
    #       SWA weight averaging bookkeeping.
    inner_model = _unwrap_to_backbone(model)

    # Target the last block stage (backbone.blocks[-1]) instead of the last
    # individual Conv2d.  Reasons:
    #   1. The last depthwise Conv2d in EfficientNetV2 produces all-negative
    #      gradients per channel — hooking it yielded blank CAM maps.
    #   2. backbone.blocks[-1] is the final Sequential stage; its output carries
    #      rich semantic features with adequate spatial resolution before the
    #      global-pooling head, giving reliable gradient flow for Grad-CAM++.
    #   3. Fallbacks are retained for architectures that lack a .blocks attribute.
    target_layer = None
    if hasattr(inner_model.backbone, 'blocks') and len(inner_model.backbone.blocks) > 0:
        target_layer = inner_model.backbone.blocks[-1]
        print(f"  [GradCAM] Target: backbone.blocks[-1] ({type(target_layer).__name__})")

    if target_layer is None:
        # Fallback: last Conv2d with spatial kernel (non-EfficientNet backbones)
        for module in inner_model.backbone.modules():
            if isinstance(module, nn.Conv2d) and module.kernel_size[0] > 1:
                target_layer = module
        if target_layer is not None:
            print(f"  [GradCAM] Fallback target: last spatial Conv2d (kernel>1)")

    if target_layer is None:
        # Final fallback: absolute last Conv2d
        for module in inner_model.backbone.modules():
            if isinstance(module, nn.Conv2d):
                target_layer = module
        print(f"  [GradCAM] Final fallback: absolute last Conv2d")

    gcam = GradCAMPlusPlusPyTorch(inner_model, target_layer)
    tfm = get_transforms(augment=False)

    fig, axes = plt.subplots(Config.NUM_CLASSES, 5, figsize=(18, 4*Config.NUM_CLASSES))
    cols = ["Original", "SHAP", "Grad-CAM++", "IG", "Ensemble (IoU)"]
    for j, t in enumerate(cols):
        axes[0][j].set_title(t, fontweight="bold")

    for ci in range(Config.NUM_CLASSES):
        idx = np.where(y_test_int == ci)[0]
        if not len(idx): continue
        img = X_test[idx[0]]
        img_tensor = tfm(img).unsqueeze(0).to(DEVICE)

        gc_map = gcam(img_tensor, ci)
        # Pass inner_model so IG gradients are not scaled by temperature
        ig_map = compute_integrated_gradients_pytorch(inner_model, img_tensor, ci)

        sh_raw = shap_results.get(ci, np.zeros((3, 224, 224)))
        sh_map = np.abs(sh_raw)
        # Reduce any extra dimensions (channels and/or classes) to spatial (H, W)
        while sh_map.ndim > 2:
            sh_map = sh_map.mean(axis=0)
        
        sh_map = sh_map / (sh_map.max() + 1e-8)

        ens = (sh_map + gc_map + ig_map) / 3.0
        iou = compute_xai_agreement_pytorch(sh_map, gc_map, ig_map)
        # Persist per-class IoU to module-level dict for JSON summary (Comment 8).
        _XAI_IOU_PER_CLASS[Config.CLASS_LABELS[ci]] = {
            "shap_vs_gradcam": float(iou["SG"]),
            "shap_vs_ig":      float(iou["SI"]),
            "gradcam_vs_ig":   float(iou["GI"]),
            "mean":            float(iou["mean"]),
        }

        row = axes[ci]
        row[0].imshow(img.astype(float)/255.0 if img.max() > 1 else img)
        row[0].set_ylabel(Config.CLASS_LABELS[ci], fontweight="bold")
        row[1].imshow(sh_map, cmap="bwr")
        row[2].imshow(gc_map, cmap="jet")
        row[3].imshow(ig_map, cmap="hot")
        row[4].imshow(ens, cmap="viridis")
        txt = f"S-G: {iou['SG']:.3f}\nS-I: {iou['SI']:.3f}\nG-I: {iou['GI']:.3f}\nMean: {iou['mean']:.3f}"
        row[4].text(5, 50, txt, color="white", fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.6))
        for ax in row:
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, "xai_ensemble_pytorch.pdf"), bbox_inches='tight')
    plt.close()
    gc.collect()


def main():
    # Ensure output directory exists before logging
    os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
    sys.stdout = Logger(os.path.join(Config.OUTPUT_DIR, "experiment_results.txt"))
    
    print("\n" + "="*70)
    print("  BRAIN TUMOR MRI CLASSIFICATION – PyTorch")
    print("="*70)

    # ── 1. Data ────────────────────────────────────────────────────────────
    # NOTE FOR REVIEWERS: All released images are first pooled, screened with
    # perceptual hashing, grouped when exact/near duplicates are detected, and
    # then split at the duplicate-group level. This prevents pHash duplicate
    # groups from crossing train/cal/val/test partitions. All baselines are
    # trained and evaluated on the identical leakage-safe partitions.
    X_tr, y_tr, X_cal, y_cal, X_va, y_va, X_te, y_te, le = load_dataset_unified_pytorch(
        Config.TRAIN_DIR, Config.TEST_DIR)

    # The dedicated calibration split is created inside load_dataset_unified_pytorch()
    # using the same pHash duplicate groups. This prevents duplicate or near-duplicate
    # images from crossing train/cal/val/test partitions.
    print(f"  [DATA] Train: {len(X_tr)}  Val: {len(X_va)}  Cal: {len(X_cal)}  Test: {len(X_te)}")

    # ── 2. Cross-Validation ────────────────────────────────────────────────
    cv_results = cross_validate_model_pytorch(X_tr, y_tr)

    # ── 3. Ablation Study ─────────────────────────────────────────────────
    ablation_results = ablation_study_pytorch(X_tr, y_tr, X_va, y_va, X_te, y_te, le)

    # ── 4. Final Model Training ───────────────────────────────────────────
    print("\n" + "="*60)
    print("  FINAL MODEL TRAINING")
    print("="*60)
    model = BrainTumorModel().to(DEVICE)
    # train_model_pytorch returns the SWA-averaged model when USE_SWA=True,
    # otherwise the best-checkpoint model.  Either way `model` is the final
    # inference model and must be the one wrapped by TemperatureScaler so
    # that calibration and evaluation are always consistent.
    history, model = train_model_pytorch(model, X_tr, y_tr, X_va, y_va)

    # ── 5. Performance Evaluation (With Calibration Comparison) ───────────
    test_loader = DataLoader(
        BrainTumorDataset(X_te, y_te, transform=get_transforms(augment=False)),
        batch_size=Config.BATCH_SIZE, shuffle=False)
        
    # Evaluate BEFORE Calibration
    print("\n  [EVAL] Evaluating baseline performance (before calibration)...")
    y_pred_uncal, y_prob_uncal, cm_uncal, stats_uncal = evaluate_model_pytorch(
        model, test_loader, DEVICE, label="Proposed (Uncalibrated)")
        
    # Run Calibration on the DEDICATED CALIBRATION SPLIT (X_cal / y_cal),
    # carved earlier from training. Comment 6 fix: avoids double-dipping the
    # validation set (which was used for early-stopping decisions during training).
    cal_loader = DataLoader(
        BrainTumorDataset(X_cal, y_cal, transform=get_transforms(augment=False)),
        batch_size=Config.BATCH_SIZE, shuffle=False)

    calibrator = TemperatureScaler(model).to(DEVICE)
    opt_t = calibrator.set_temperature(cal_loader, DEVICE)
    
    # Evaluate AFTER Calibration (using the calibrator as a wrapped model)
    print("  [EVAL] Evaluating calibrated performance...")
    y_pred, y_prob, cm, final_metrics = evaluate_model_pytorch(
        calibrator, test_loader, DEVICE, label="Proposed (Calibrated)")
        
    # Comparison Visualization
    plot_calibration_comparison_pytorch(y_te, y_prob_uncal, y_prob, label="EffNetV2_Torch")
    
    print("\n" + classification_report(
        y_te, y_pred,
        labels=list(range(Config.NUM_CLASSES)),
        target_names=Config.CLASS_LABELS,
        zero_division=0))

    # ── 6. Plots ──────────────────────────────────────────────────────────
    plot_learning_curves(history, "D2")
    plot_confusion_matrix_pytorch(cm, "D2")
    plot_roc_curves_pytorch(y_te, y_prob, "D2")

    # ── 7. Calibration & Bootstrap ────────────────────────────────────────
    # Compute ECE for BOTH the uncalibrated and calibrated predictions so the
    # summary JSON can report the reduction.
    ece_uncal = compute_ece_and_reliability_diagram(y_te, y_prob_uncal, "EffNetV2_Torch_Uncal")
    ece = compute_ece_and_reliability_diagram(y_te, y_prob, "EffNetV2_Torch")
    print(f"\n  [ECE] Uncalibrated: {ece_uncal:.4f}   Calibrated: {ece:.4f}   "
          f"Reduction: {100*(1 - ece/max(ece_uncal,1e-12)):.1f}%")

    # Capture bootstrap CIs as a returned dict instead of just printing them.
    bootstrap_ci_dict = bootstrap_confidence_intervals_returning(
        y_te, y_pred, y_prob,
        num_classes=Config.NUM_CLASSES,
        B=Config.BOOTSTRAP_B,
        seed=SEED,
        calculate_brier_score_fn=calculate_brier_score,
        compute_multiclass_specificity_fn=compute_multiclass_specificity,
    )
    bootstrap_confidence_intervals(y_te, y_pred, y_prob)  # keep original printing too

    # ── 8. Baseline Comparison ────────────────────────────────────────────
    baseline_results = run_baseline_comparison_pytorch(X_tr, y_tr, X_va, y_va, X_te, y_te, y_pred, y_prob)

    # ── 9. XAI Suite ──────────────────────────────────────────────────────
    # Pass the calibrator so SHAP sees temperature-scaled probabilities
    # (consistent with reported metrics), while Grad-CAM++ and IG internally
    # unwrap to inner_model so attribution gradients are unaffected by T.
    shap_res = explain_with_shap_pytorch(calibrator, X_te, y_te)
    plot_xai_ensemble_pytorch(calibrator, X_te, y_te, shap_res)
    plot_deletion_insertion_pytorch(calibrator, X_te, y_te, shap_res, temperature=1.0)

    # ── 10. Save Final Model ──────────────────────────────────────────────
    # Save both the raw backbone weights and the full calibrated wrapper so
    # either can be loaded for inference.
    final_path = os.path.join(Config.OUTPUT_DIR, "final_model_v5_pytorch.pth")
    torch.save(model.state_dict(), final_path)
    print(f"\n  [SAVED] Backbone weights → {final_path}")

    calibrated_path = os.path.join(Config.OUTPUT_DIR, "final_model_v5_calibrated_pytorch.pth")
    torch.save({
        "model_state_dict": model.state_dict(),
        "temperature": calibrator._safe_temperature().item(),
    }, calibrated_path)
    print(f"  [SAVED] Calibrated model (weights + T) → {calibrated_path}")

    # 11. Experiment Summary
    final_comp = compute_computational_complexity(model)
    print("\n" + "="*60)
    print("  EXPERIMENT SUMMARY")
    print("="*60)
    print(f"  CV  Accuracy  : {np.mean(cv_results)*100:.2f}% ± {np.std(cv_results)*100:.2f}%")
    print(f"  Test Accuracy : {final_metrics['acc']*100:.2f}%")
    print(f"  Precision     : {final_metrics['prec']*100:.2f}%")
    print(f"  Recall        : {final_metrics['rec']*100:.2f}%")
    print(f"  Specificity   : {final_metrics['spec']*100:.2f}%")
    print(f"  F1-Score      : {final_metrics['f1']*100:.2f}%")
    print(f"  AUC           : {final_metrics['auc']:.4f}")
    print(f"  Kappa         : {final_metrics['kappa']:.4f}")
    print(f"  Brier Score   : {final_metrics['brier']:.4f}")
    print(f"  ECE           : {ece:.4f}")
    print(f"  Params (M)    : {final_comp['params']:.2f}M")
    print(f"  FLOPs (G)     : {final_comp['flops']:.3f}G")
    print(f"  Inference     : {final_comp['infer_time']:.2f} ms/img")
    print(f"  Temperature T : {opt_t:.4f} (Calibrated)")
    print(f"  Output Dir    : {Config.OUTPUT_DIR}/")
    print("="*60)

    # Every numerical claim in the manuscript should be traceable to a key in
    # this JSON. NEVER manually retype numbers from the printed log into a
    # paper table; regenerate from this file.
    from collections import Counter as _Counter
    try:
        _selective = selective_prediction_analysis(y_te, y_pred, y_prob)
    except Exception as _e:
        print(f"  [WARN] selective_prediction_analysis failed: {_e}")
        _selective = None

    _payload = build_summary_payload(
        dataset_key=DATASET,
        dataset_name=_dataset_cfg_entry["name"],
        seed=SEED,
        n_train=len(X_tr), n_val=len(X_va),
        n_cal=len(X_cal), n_test=len(X_te),
        per_class_support_test=dict(_Counter(y_te.tolist())),
        label_encoder=le,
        cv_acc_mean=float(np.mean(cv_results)) if len(cv_results) else 0.0,
        cv_acc_std=float(np.std(cv_results)) if len(cv_results) else 0.0,
        cv_fold_accs=[float(a) for a in cv_results],
        test_metrics_calibrated={k: float(v) for k, v in final_metrics.items()
                                  if isinstance(v, (int, float, np.floating))},
        test_metrics_uncalibrated={k: float(v) for k, v in stats_uncal.items()
                                    if isinstance(v, (int, float, np.floating))},
        bootstrap_cis_calibrated=bootstrap_ci_dict,
        bootstrap_cis_baselines=dict(_BASELINE_BOOTSTRAP_CIS) or None,
        ece_uncalibrated=float(ece_uncal),
        ece_calibrated=float(ece),
        temperature=float(opt_t),
        params_M=float(final_comp["params"]),
        flops_G=float(final_comp["flops"]),
        inference_ms=float(final_comp["infer_time"]),
        ablation_rows=list(_ABLATION_RESULTS),
        mcnemar_rows=list(_MCNEMAR_ROWS_ABLATION) + list(_MCNEMAR_ROWS_BASELINE),
        xai_iou_per_class=dict(_XAI_IOU_PER_CLASS),
        xai_iou_random_baseline=float(compute_random_iou_baseline(Config.XAI_TOP_K_PCT)),
        deletion_insertion_auc=dict(_XAI_DEL_INS_AUC),
        selective_prediction=_selective,
        duplicate_detection_phash=None,
        duplicate_detection_deep=None,
        extra={
            "baseline_results": list(_BASELINE_RESULTS),
            "baseline_predictions_meta": dict(_BASELINE_PREDICTIONS),
            "smoke_test": SMOKE_TEST,
            "epochs_used": Config.EPOCHS,
            "n_folds_used": Config.N_FOLDS,
            "label_smoothing": Config.LABEL_SMOOTHING,
            "augmentation_rotation_deg": Config.AUG_ROTATION,
        },
    )
    save_experiment_summary(Config.OUTPUT_DIR, _payload)


if __name__ == "__main__":
    main()